# Solar Prediction Colab Horizon Sweep

Run the cleaned package on the full ignored dataset at several forecast horizons. This makes the persistence baseline less trivial than a one-step 15-minute forecast.

Expected data path inside the uploaded Drive project: `data/solar_weather.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your Drive folder name differs.
PROJECT_DIR = '/content/drive/MyDrive/solar_prediction'
DATA_PATH = 'data/solar_weather.csv'
OUTPUT_ROOT = 'artifacts/colab_horizon_sweep_actual_data'

EPOCHS = 50
HIDDEN_DIM = 32
BATCH_SIZE = 256
SEASONAL_LAG = 96  # Daily cycle for 15-minute data.
HORIZONS = {
    '15min': 1,
    '1h': 4,
    '4h': 16,
    '24h': 96,
}


In [ ]:
import os
from pathlib import Path

project = Path(PROJECT_DIR)
assert project.exists(), f'Project folder not found: {project}'
os.chdir(project)

data_path = Path(DATA_PATH)
assert data_path.exists(), f'Full dataset not found: {data_path}'

print('Working directory:', Path.cwd())
print('Full dataset:', data_path)


If an earlier Colab session mixed incompatible NumPy/SciPy/pandas binaries, restart the runtime and run from the top.

In [ ]:
%pip install -q --upgrade --force-reinstall "numpy==2.0.2" "pandas==2.2.2" "scipy==1.14.1" "scikit-learn==1.6.1"
%pip install -q -e .


In [ ]:
import pandas as pd

preview = pd.read_csv(data_path, nrows=5)
time_col = pd.read_csv(data_path, usecols=['Time'])['Time']
parsed_time = pd.to_datetime(time_col, errors='coerce')

print('Rows:', len(time_col))
print('Columns:', preview.columns.tolist())
print('Time range:', parsed_time.min(), 'to', parsed_time.max())
preview.head()


In [ ]:
from io import StringIO
import shlex
import subprocess

import pandas as pd
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
all_metrics = []

for label, horizon_steps in HORIZONS.items():
    output_dir = f'{OUTPUT_ROOT}/{label}'
    cmd = [
        'solar-predict',
        'compare',
        '--data', str(data_path),
        '--epochs', str(EPOCHS),
        '--hidden-dim', str(HIDDEN_DIM),
        '--batch-size', str(BATCH_SIZE),
        '--horizon-steps', str(horizon_steps),
        '--seasonal-lag', str(SEASONAL_LAG),
        '--device', device,
        '--output', output_dir,
        '--quiet',
    ]

    print('\nHorizon:', label, f'({horizon_steps} steps)')
    print('Command:', shlex.join(cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')

    output_lines = result.stdout.splitlines()
    header_idx = next(i for i, line in enumerate(output_lines) if line.startswith('model,'))
    metrics = pd.read_csv(StringIO('\n'.join(output_lines[header_idx:])))
    metrics.insert(0, 'horizon', label)
    metrics.insert(1, 'horizon_steps', horizon_steps)
    all_metrics.append(metrics)

sweep_metrics = pd.concat(all_metrics, ignore_index=True)
output_root = Path(OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)
sweep_metrics.to_csv(output_root / 'horizon_sweep_metrics.csv', index=False)
sweep_metrics


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
horizon_order = list(HORIZONS)
rmse_table = sweep_metrics.pivot(index='horizon', columns='model', values='rmse').loc[horizon_order]
r2_table = sweep_metrics.pivot(index='horizon', columns='model', values='r2').loc[horizon_order]

rmse_table.plot.bar(ax=axes[0])
axes[0].set_title('RMSE by horizon')
axes[0].set_xlabel('')
axes[0].set_ylabel('W/m^2')

r2_table.plot.bar(ax=axes[1])
axes[1].set_title('R2 by horizon')
axes[1].set_xlabel('')
axes[1].set_ylabel('')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

best_by_horizon = sweep_metrics.sort_values('rmse').groupby('horizon', sort=False).first()
best_by_horizon[['model', 'rmse', 'mae', 'r2', 'capped_mape']]
